# Encrypted Finance Tutorial: Core (`finance/core.py`)

This tutorial covers `src/concrete_fhe_toolkit/finance/core.py`. In FHE, floats are restricted. This module provides a way to calculate rates, taxes, discounts, and interests using an integer `RATE_SCALE` (which is 100). All encrypted money is represented in cents/kurus.

## 1. Applying Rates and Taxes

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.finance.core import apply_rate, return_actual_value, calculate_tax

def test_finance_core(amount: int):
    # Apply a 5% rate and an 18% tax
    rate_applied = apply_rate(amount, rate=0.05)
    tax_applied = calculate_tax(amount, rate=0.18)
    return rate_applied, tax_applied

compiler = fhe.Compiler(test_finance_core, {"amount": "encrypted"})
inputset = [(1000,), (5000,)]
circuit = compiler.compile(inputset)

# Amount: 1000 cents (10.00 dollars)
enc_rate, enc_tax = circuit.encrypt_run_decrypt(1000)

# Decode back to real values (divides by RATE_SCALE=100)
real_rate = return_actual_value(enc_rate)
real_tax = return_actual_value(enc_tax)

assert real_rate == 50.0  # 5% of 1000 is 50
assert real_tax == 180.0  # 18% of 1000 is 180
print("✅ Encrypted rates and taxes passed!")

## 2. Discounts and Simple Interest

In [ ]:
from concrete_fhe_toolkit.finance.core import discount, simple_interest

def test_discount_interest(amount: int, time: int):
    disc = discount(amount, rate=0.20)
    interest = simple_interest(amount, rate=0.05, time_period=time)
    return disc, interest

compiler = fhe.Compiler(test_discount_interest, {"amount": "encrypted", "time": "encrypted"})
inputset = [(1000, 1), (5000, 3)]
circuit = compiler.compile(inputset)

enc_disc, enc_int = circuit.encrypt_run_decrypt(1000, 3)

real_disc = return_actual_value(enc_disc)
real_int = return_actual_value(enc_int)

assert real_disc == 800.0  # 1000 with 20% discount
assert real_int == 150.0  # 1000 * 5% * 3 years
print("✅ Encrypted discounts and interests passed!")